# Operations 06 Log to File (LangChain 2026)

## What This Lesson Is
Persist structured call logs for post-run diagnostics.

## Scientific Lens
- Concept: Structured execution logging
- Measure: Log completeness and parse success
- Validity Limit: File logs alone are insufficient for fleet-wide observability.


## How It Works
1. Define log schema.
2. Write/read deterministic entries.
3. Emit live callback logs to file.


In [ ]:
print("Log-to-file lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: real provider/tool path with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
from pathlib import Path
import json
p=Path("/tmp/langchain_ops_log.jsonl")
p.write_text(json.dumps({"event":"invoke","ok":True})+"\n")
print(p.read_text())
assert "invoke" in p.read_text()


In [ ]:
# Live Demo
import os, json
from pathlib import Path
from langchain.callbacks.base import BaseCallbackHandler

class FileTrace(BaseCallbackHandler):
    def __init__(self, path: Path):
        self.path = path
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.path.open('a').write(json.dumps({'event':'start','n':len(prompts)}) + '\\n')
    def on_llm_end(self, response, **kwargs):
        self.path.open('a').write(json.dumps({'event':'end'}) + '\\n')

try:
    from langchain_openai import ChatOpenAI
except Exception as exc:
    print(f"Skipping live logging run: dependency unavailable ({exc})")
else:
    if not os.getenv("OPENAI_API_KEY"):
        print("Skipping live logging run: OPENAI_API_KEY not set.")
    else:
        path = Path('/tmp/langchain_live_trace.jsonl')
        llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0, callbacks=[FileTrace(path)], timeout=20)
        out = llm.invoke('One sentence: why structured logs help incident response?')
        print(out.content if hasattr(out, 'content') else out)
        print(path.read_text())


## Applied Labs
1. Add correlation IDs to each log event.
2. Rotate log file on size threshold.
3. Parse logs into call count and failure summaries.

## Validation Checklist
- Logs are JSON-parseable lines.
- Start/end events appear for successful run.
- Log path is writable in container context.

## Further Reading
- [LangChain callbacks](https://python.langchain.com/docs/concepts/callbacks/)
- [JSON Lines](https://jsonlines.org/)
- [OpenTelemetry logs](https://opentelemetry.io/docs/specs/otel/logs/)
